# Adversarial Evaluation Notebook for the ResNet-50 Shoulder Classifier

This notebook mirrors the existing training pipeline but focuses on loading a
pretrained model checkpoint (e.g. the one obtained after 500 epochs) and
running adversarial attack experiments. The notebook keeps the original split
between training and validation sets—**no additional split is introduced**—so
that the distributions remain consistent with the training script.

## 1. Configuration and Paths

Update `DATA_ROOT` so that it points to the folder that contains the
`MURA-v1.1/` directory on your machine. The `RUN_DIR` should match the directory
that already stores your `checkpoints/`, `logs.txt`, and the best model
(`resnet50_best.pt`).

In [ ]:
import os
import json
import math
import random
from collections import Counter, defaultdict
from pathlib import Path
from typing import Dict, List, Tuple

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms

# -----------------------------------------------------------------------------
# Reproduce the exact directory layout used by the training script.
# -----------------------------------------------------------------------------
DATA_ROOT = Path(r"C:/Users/NaralaG3762/Downloads/MURA-v1.1/MURA-v1.1").expanduser()
ANATOMY_SUBSET = None           # e.g. "XR_SHOULDER" to focus on a single anatomy
RUN_DIR = DATA_ROOT / "AA_v3_multi_models_dml_runs"
MODEL_NAME = "resnet50"
IMG_SIZE = 320
BATCH_SIZE = 24
NUM_CLASSES = 2
SEED = 42

# Checkpoint locations – these should already exist from the previous 500-epoch run.
CHECKPOINT_DIR = RUN_DIR / "checkpoints"
LATEST_CKPT = CHECKPOINT_DIR / "latest.pt"
BEST_PATH = RUN_DIR / f"{MODEL_NAME}_best.pt"
HISTORY_JSON = CHECKPOINT_DIR / "history.json"

# If you want to resume exactly where you left off, keep this True.
RESUME = True

# Imagenet normalization constants used during training.
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

## 2. Reproducibility and Device Selection

We re-use the device selection helper from the training script so that DirectML
is preferred when available, followed by CUDA, then CPU. Setting all random
seeds keeps augmentations and adversarial perturbations deterministic between
runs.

In [ ]:
# -----------------------------------------------------------------------------
# Seed everything for deterministic behaviour.
# -----------------------------------------------------------------------------
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# -----------------------------------------------------------------------------
# Select the same device hierarchy as the training script.
# -----------------------------------------------------------------------------
def pick_device(prefer_gpu: bool = True) -> torch.device:
    if prefer_gpu:
        try:
            import torch_directml
            device = torch_directml.device()
            _ = torch.randn(1, device=device)
            print(f"Using DirectML device: {device}")
            return device
        except Exception:
            pass
        if torch.cuda.is_available():
            print(f"Using CUDA GPU: {torch.cuda.get_device_name(0)}")
            return torch.device("cuda")
    print("Falling back to CPU")
    return torch.device("cpu")

DEVICE = pick_device(prefer_gpu=True)

## 3. Dataset Indexing (No Additional Split)

The helper below scans the `train/` and `valid/` folders from the official
MURA-v1.1 release. This is the exact split used during training, so no new
70/20/10 division is introduced. Hidden AppleDouble files and missing images are
filtered out in the same way as the training script.

In [ ]:
VALID_EXTS = {".png", ".jpg", ".jpeg"}


def scan_mura_split(split_root: Path, anatomy_subset: str | None = None) -> pd.DataFrame:
    rows: List[Dict[str, int]] = []
    for anatomy_dir in sorted(split_root.iterdir()):
        if not anatomy_dir.is_dir():
            continue
        if anatomy_subset and anatomy_dir.name != anatomy_subset:
            continue
        for patient_dir in anatomy_dir.iterdir():
            if not patient_dir.is_dir():
                continue
            for study_dir in patient_dir.iterdir():
                if not study_dir.is_dir():
                    continue
                label = 1 if "positive" in study_dir.name.lower() else 0
                for file_path in study_dir.iterdir():
                    if file_path.suffix.lower() in VALID_EXTS:
                        rows.append({"path": str(file_path), "label": label})
    return pd.DataFrame(rows)


def clean_df(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    keep_mask = []
    repaired = 0
    dropped = 0
    for path in df["path"].astype(str):
        file_path = Path(path)
        name = file_path.name
        suffix = file_path.suffix.lower()
        if name.startswith(".") or suffix not in VALID_EXTS:
            if name.startswith("._"):
                alt = file_path.parent / name[2:]
                if alt.is_file():
                    keep_mask.append(str(alt))
                    repaired += 1
                    continue
            dropped += 1
            keep_mask.append(None)
            continue
        if file_path.is_file():
            keep_mask.append(path)
        else:
            if name.startswith("._"):
                alt = file_path.parent / name[2:]
                if alt.is_file():
                    keep_mask.append(str(alt))
                    repaired += 1
                    continue
            keep_mask.append(None)
            dropped += 1
    if repaired or dropped:
        print(f"[clean_df] repaired={repaired} dropped={dropped}")
    df["path"] = keep_mask
    df = df.dropna(subset=["path"]).reset_index(drop=True)
    df["path"] = df["path"].astype(str)
    df["label"] = df["label"].astype(int)
    return df


train_root = DATA_ROOT / "train"
valid_root = DATA_ROOT / "valid"
if not train_root.exists() or not valid_root.exists():
    raise FileNotFoundError(
        "The expected MURA train/valid folders were not found. "
        "Double-check that DATA_ROOT points to the directory that contains "
        "the 'train' and 'valid' folders."
    )

train_df = clean_df(scan_mura_split(train_root, anatomy_subset=ANATOMY_SUBSET))
valid_df = clean_df(scan_mura_split(valid_root, anatomy_subset=ANATOMY_SUBSET))

print(f"Train images: {len(train_df)}")
print(f"Valid images: {len(valid_df)}")
print("Label dist (train):", Counter(train_df["label"]))
print("Label dist (valid):", Counter(valid_df["label"]))

## 4. Classical Preprocessing Pipeline

We reuse the exact windowing → CLAHE → edge boost → auto-crop routine so that
inference remains consistent with the training setup. Each transform contains
inline documentation explaining its effect.

In [ ]:
class BoneWindowTransform:
    '''Percentile-based intensity windowing to emphasise bone structure.'''

    def __init__(self, lower_percentile: float = 3, upper_percentile: float = 97) -> None:
        self.lower = lower_percentile
        self.upper = upper_percentile

    def __call__(self, img: np.ndarray) -> np.ndarray:
        arr = img.astype(np.float32)
        non_zero = arr[arr > 0]
        if non_zero.size == 0:
            return img
        low = np.percentile(non_zero, self.lower)
        high = np.percentile(non_zero, self.upper)
        if high <= low:
            return img
        clipped = np.clip(arr, low, high)
        scaled = (clipped - low) / (high - low)
        return (scaled * 255.0).astype(np.uint8)


class ClaheTransform:
    '''Contrast Limited Adaptive Histogram Equalisation (CLAHE).'''

    def __init__(self, clip_limit: float = 2.5, tile_grid_size: Tuple[int, int] = (8, 8)) -> None:
        self.clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid_size)

    def __call__(self, img: np.ndarray) -> np.ndarray:
        return self.clahe.apply(img.astype(np.uint8))


class EdgeTransform:
    '''Adds a scaled Sobel gradient magnitude to highlight boundaries.'''

    def __init__(self, alpha: float = 0.5) -> None:
        self.alpha = float(alpha)

    def __call__(self, img: np.ndarray) -> np.ndarray:
        base = img.astype(np.float32)
        gx = cv2.Sobel(base, cv2.CV_32F, 1, 0, ksize=3)
        gy = cv2.Sobel(base, cv2.CV_32F, 0, 1, ksize=3)
        mag = np.sqrt(gx * gx + gy * gy)
        mag = (mag / (mag.max() + 1e-6) * 255.0)
        mixed = np.clip(base + self.alpha * mag, 0, 255)
        return mixed.astype(np.uint8)


class AutoCropTransform:
    '''Crops to the largest foreground component if present.'''

    def __init__(self, thresh: int = 5) -> None:
        self.thresh = thresh

    def __call__(self, img: np.ndarray) -> np.ndarray:
        _, binary = cv2.threshold(img, self.thresh, 255, cv2.THRESH_BINARY)
        contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if not contours:
            return img
        x, y, w, h = cv2.boundingRect(max(contours, key=cv2.contourArea))
        if w < 16 or h < 16:
            return img
        return img[y : y + h, x : x + w]


window = BoneWindowTransform(3, 97)
clahe = ClaheTransform(2.5, (8, 8))
edge = EdgeTransform(0.5)
autocrop = AutoCropTransform()


def preprocess_chain(img: np.ndarray) -> np.ndarray:
    '''Sequentially apply all preprocessing steps.'''

    out = window(img)
    out = clahe(out)
    out = edge(out)
    out = autocrop(out)
    return out

## 5. Dataset and DataLoader

`MuraDataset` mirrors the training version: it applies preprocessing on-the-fly
and uses identical augmentations for the training split. Validation samples are
only resized and normalised.

In [ ]:
train_tfms = transforms.Compose([
    transforms.ToPILImage(),
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.9, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

valid_tfms = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])


class MuraDataset(Dataset):
    def __init__(self, df: pd.DataFrame, augment: bool = False) -> None:
        self.paths: List[str] = df["path"].tolist()
        self.labels: List[int] = df["label"].astype(int).tolist()
        self.augment = augment

    def __len__(self) -> int:  # type: ignore[override]
        return len(self.paths)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, int]:  # type: ignore[override]
        path = self.paths[idx]
        label = self.labels[idx]
        img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
        if img is None:
            raise FileNotFoundError(path)
        img = preprocess_chain(img)
        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)
        img = np.stack([img, img, img], axis=-1)
        tfm = train_tfms if self.augment else valid_tfms
        tensor = tfm(img)
        return tensor, label


train_ds = MuraDataset(train_df, augment=True)
valid_ds = MuraDataset(valid_df, augment=False)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
valid_loader = DataLoader(valid_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

## 6. Model Definition and Checkpoint Loading

The notebook constructs a ResNet-50 (or DenseNet-121 if you change the
configuration) with the same heads as during training. `try_load_checkpoint`
mirrors the training helper and restores weights, optimizer, scheduler, and RNG
state so that you can continue training if desired.

In [ ]:
def build_model(name: str = "resnet50", num_classes: int = 2) -> nn.Module:
    name = name.lower()
    if name == "resnet50":
        model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
        model.fc = nn.Linear(model.fc.in_features, num_classes)
    elif name == "densenet121":
        model = models.densenet121(weights=models.DenseNet121_Weights.IMAGENET1K_V1)
        model.classifier = nn.Linear(model.classifier.in_features, num_classes)
    else:
        raise ValueError("Supported models: resnet50, densenet121")
    return model


model = build_model(MODEL_NAME, NUM_CLASSES).to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=3e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=1)


def get_rng_state() -> Dict[str, object]:
    return {
        "py_random": random.getstate(),
        "np_random": np.random.get_state(),
        "torch_cpu": torch.get_rng_state().tolist(),
        "torch_cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,
    }


def set_rng_state(state: Dict[str, object]) -> None:
    try:
        random.setstate(state["py_random"])
    except Exception:
        pass
    try:
        np.random.set_state(state["np_random"])
    except Exception:
        pass
    try:
        torch.set_rng_state(torch.tensor(state["torch_cpu"], dtype=torch.uint8))
    except Exception:
        pass
    if torch.cuda.is_available() and state.get("torch_cuda") is not None:
        try:
            torch.cuda.set_rng_state_all(state["torch_cuda"])
        except Exception:
            pass


def try_load_checkpoint(
    source: str | int | Path | None = "latest",
    *,
    load_optimizer: bool = True,
    load_scheduler: bool = True,
    load_rng: bool = True,
    strict_model_name: bool = True,
) -> Tuple[int, float, dict] | None:
    if source in (None, "latest"):
        ckpt_path = LATEST_CKPT
    elif isinstance(source, int):
        ckpt_path = CHECKPOINT_DIR / f"epoch_{source:03d}" / f"epoch_{source:03d}.pt"
    else:
        ckpt_path = Path(source)
    if not ckpt_path.exists():
        print(f"[warn] Checkpoint not found: {ckpt_path}")
        return None
    checkpoint = torch.load(ckpt_path, map_location=DEVICE)
    src_name = str(checkpoint.get("model_name", "")).lower()
    if strict_model_name and src_name != MODEL_NAME.lower():
        print(f"[warn] Model mismatch: saved '{src_name}' vs current '{MODEL_NAME}'")
        return None
    model.load_state_dict(checkpoint["model_state"])
    if load_optimizer and "optimizer_state" in checkpoint:
        try:
            optimizer.load_state_dict(checkpoint["optimizer_state"])
        except Exception as exc:
            print(f"[warn] Unable to restore optimizer: {exc}")
    if load_scheduler and scheduler is not None and checkpoint.get("scheduler_state") is not None:
        try:
            scheduler.load_state_dict(checkpoint["scheduler_state"])
        except Exception as exc:
            print(f"[warn] Unable to restore scheduler: {exc}")
    if load_rng and checkpoint.get("rng_state") is not None:
        set_rng_state(checkpoint["rng_state"])
    start_epoch = int(checkpoint.get("epoch", 0)) + 1
    best_val = float(checkpoint.get("best_val", math.inf))
    history = checkpoint.get("history", defaultdict(list))
    if isinstance(history, defaultdict):
        history = dict(history)
    print(f"Loaded checkpoint '{ckpt_path}' (next epoch: {start_epoch}, best val: {best_val:.4f})")
    return start_epoch, best_val, history


resume_info = try_load_checkpoint("latest") if RESUME else None

## 7. Baseline Validation Accuracy

Before launching attacks we measure standard validation performance to verify
that the checkpoint was restored correctly.

In [ ]:
@torch.no_grad()
def evaluate(loader: DataLoader, max_batches: int | None = None) -> Dict[str, float]:
    model.eval()
    total = 0
    correct = 0
    loss_sum = 0.0
    for batch_idx, (xb, yb) in enumerate(loader):
        xb = xb.to(DEVICE)
        yb = yb.to(DEVICE)
        logits = model(xb)
        loss = criterion(logits, yb)
        preds = logits.argmax(dim=1)
        total += yb.size(0)
        correct += (preds == yb).sum().item()
        loss_sum += loss.item() * yb.size(0)
        if max_batches is not None and batch_idx + 1 >= max_batches:
            break
    return {
        "loss": loss_sum / max(1, total),
        "acc": correct / max(1, total),
        "samples": total,
    }


baseline_metrics = evaluate(valid_loader)
baseline_metrics

## 8. Adversarial Attack Utilities

We implement three common attacks:

* **FGSM** – single-step Fast Gradient Sign Method.
* **PGD** – multi-step Projected Gradient Descent with clamped perturbations.
* **DeepFool** – untargeted iterative attack that estimates the decision
  boundary (CPU friendly version for small batches).

All attacks respect the same preprocessing pipeline by perturbing *normalised*
tensors and then re-normalising/clipping appropriately.

In [ ]:
from dataclasses import dataclass


def clamp_tensor(t: torch.Tensor, lower: float = 0.0, upper: float = 1.0) -> torch.Tensor:
    lower_t = torch.tensor(lower, device=t.device)
    upper_t = torch.tensor(upper, device=t.device)
    return torch.max(torch.min(t, upper_t), lower_t)


@dataclass
class AttackResult:
    inputs: torch.Tensor
    adv_inputs: torch.Tensor
    labels: torch.Tensor
    original_logits: torch.Tensor
    adv_logits: torch.Tensor


def fgsm_attack(
    inputs: torch.Tensor,
    labels: torch.Tensor,
    *,
    epsilon: float = 8 / 255,
) -> AttackResult:
    model.eval()
    inputs = inputs.clone().detach().to(DEVICE).requires_grad_(True)
    labels = labels.to(DEVICE)
    logits = model(inputs)
    loss = criterion(logits, labels)
    loss.backward()
    adv_inputs = inputs + epsilon * inputs.grad.sign()
    for c, mean, std in zip(range(3), IMAGENET_MEAN, IMAGENET_STD):
        adv_inputs[:, c, :, :] = clamp_tensor(adv_inputs[:, c, :, :], (0 - mean) / std, (1 - mean) / std)
    adv_logits = model(adv_inputs.detach())
    return AttackResult(inputs.detach(), adv_inputs.detach(), labels.detach(), logits.detach(), adv_logits.detach())


def pgd_attack(
    inputs: torch.Tensor,
    labels: torch.Tensor,
    *,
    epsilon: float = 8 / 255,
    step_size: float = 2 / 255,
    steps: int = 10,
) -> AttackResult:
    model.eval()
    labels = labels.to(DEVICE)
    inputs = inputs.clone().detach().to(DEVICE)
    adv = inputs + torch.empty_like(inputs).uniform_(-epsilon, epsilon)
    for c, mean, std in zip(range(3), IMAGENET_MEAN, IMAGENET_STD):
        adv[:, c, :, :] = clamp_tensor(adv[:, c, :, :], (0 - mean) / std, (1 - mean) / std)
    adv.requires_grad_(True)
    for _ in range(steps):
        logits = model(adv)
        loss = criterion(logits, labels)
        grad = torch.autograd.grad(loss, adv, retain_graph=False, create_graph=False)[0]
        adv = adv + step_size * grad.sign()
        delta = torch.clamp(adv - inputs, -epsilon, epsilon)
        adv = inputs + delta
        for c, mean, std in zip(range(3), IMAGENET_MEAN, IMAGENET_STD):
            adv[:, c, :, :] = clamp_tensor(adv[:, c, :, :], (0 - mean) / std, (1 - mean) / std)
        adv = adv.detach().requires_grad_(True)
    adv_logits = model(adv)
    return AttackResult(inputs.detach(), adv.detach(), labels.detach(), model(inputs).detach(), adv_logits.detach())


def deepfool_attack(
    inputs: torch.Tensor,
    labels: torch.Tensor,
    *,
    steps: int = 50,
    overshoot: float = 0.02,
    num_classes: int = NUM_CLASSES,
) -> AttackResult:
    model.eval()
    inputs = inputs.clone().detach().to(DEVICE)
    labels = labels.to(DEVICE)
    adv = inputs.clone().detach()
    adv.requires_grad_(True)
    original_logits = model(inputs)
    batch_size = inputs.size(0)
    for i in range(batch_size):
        x = adv[i : i + 1].clone().detach().requires_grad_(True)
        for _ in range(steps):
            logits = model(x)
            pred = logits.argmax(dim=1)
            if pred != labels[i]:
                break
            current = logits[0, pred]
            grad_current = torch.autograd.grad(current, x, retain_graph=True)[0]
            min_delta = math.inf
            perturbation = None
            for k in range(num_classes):
                if k == pred:
                    continue
                target_logit = logits[0, k]
                grad_target = torch.autograd.grad(target_logit, x, retain_graph=True)[0]
                w_k = grad_target - grad_current
                f_k = (target_logit - current).item()
                if w_k.norm() == 0:
                    continue
                delta_k = abs(f_k) / (w_k.norm() + 1e-8)
                if delta_k < min_delta:
                    min_delta = delta_k
                    perturbation = w_k
            if perturbation is None:
                break
            r_i = (min_delta + 1e-4) * perturbation / (perturbation.norm() + 1e-8)
            x = x + (1 + overshoot) * r_i
            for c, mean, std in zip(range(3), IMAGENET_MEAN, IMAGENET_STD):
                x[:, c, :, :] = clamp_tensor(x[:, c, :, :], (0 - mean) / std, (1 - mean) / std)
        adv[i] = x.detach()[0]
    adv_logits = model(adv)
    return AttackResult(inputs.detach(), adv.detach(), labels.detach(), original_logits.detach(), adv_logits.detach())

## 9. Running Attacks and Measuring Impact

The helper below evaluates a chosen attack on the validation set. To keep things
light-weight you can cap `max_batches`; set it to `None` to process the entire
validation loader.

In [ ]:
from tqdm import tqdm


def evaluate_attack(
    loader: DataLoader,
    attack_fn,
    *,
    max_batches: int | None = 5,
) -> Dict[str, float]:
    model.eval()
    clean_correct = 0
    adv_correct = 0
    total = 0
    for batch_idx, (xb, yb) in enumerate(tqdm(loader, desc=attack_fn.__name__)):
        xb = xb.to(DEVICE)
        yb = yb.to(DEVICE)
        with torch.no_grad():
            clean_logits = model(xb)
            clean_pred = clean_logits.argmax(dim=1)
            clean_correct += (clean_pred == yb).sum().item()
        result = attack_fn(xb, yb)
        adv_pred = result.adv_logits.argmax(dim=1)
        adv_correct += (adv_pred == yb).sum().item()
        total += yb.size(0)
        if max_batches is not None and batch_idx + 1 >= max_batches:
            break
    return {
        "clean_acc": clean_correct / max(1, total),
        "adv_acc": adv_correct / max(1, total),
        "samples": total,
    }


fgsm_metrics = evaluate_attack(valid_loader, lambda x, y: fgsm_attack(x, y, epsilon=8/255), max_batches=5)
fgsm_metrics

Repeat the evaluation for Projected Gradient Descent and DeepFool. Increase the
number of batches or adjust the epsilon/step counts as needed for your
experiments.

In [ ]:
pgd_metrics = evaluate_attack(
    valid_loader,
    lambda x, y: pgd_attack(x, y, epsilon=8/255, step_size=2/255, steps=10),
    max_batches=5,
)
pgd_metrics

In [ ]:
deepfool_metrics = evaluate_attack(valid_loader, deepfool_attack, max_batches=3)
deepfool_metrics

## 10. Visualising Adversarial Examples

The function below plots a few samples from an attack to visually inspect the
perturbations. It shows the clean and adversarial images along with their
predicted probabilities.

In [ ]:
import matplotlib.pyplot as plt


def plot_attack_examples(result: AttackResult, num_examples: int = 4) -> None:
    probs_clean = result.original_logits.softmax(dim=1)
    probs_adv = result.adv_logits.softmax(dim=1)
    for idx in range(min(num_examples, result.inputs.size(0))):
        clean = result.inputs[idx].cpu()
        adv = result.adv_inputs[idx].cpu()
        label = result.labels[idx].item()
        clean_prob = probs_clean[idx, label].item()
        adv_prob = probs_adv[idx, label].item()

        def denorm(t: torch.Tensor) -> np.ndarray:
            arr = t.clone()
            for c, mean, std in zip(range(3), IMAGENET_MEAN, IMAGENET_STD):
                arr[c] = arr[c] * std + mean
            return arr.clamp(0, 1).permute(1, 2, 0).numpy()

        fig, axes = plt.subplots(1, 3, figsize=(10, 3))
        axes[0].imshow(denorm(clean), cmap="gray")
        axes[0].set_title(f"Clean (p={clean_prob:.2f})")
        axes[0].axis("off")

        axes[1].imshow(denorm(adv), cmap="gray")
        axes[1].set_title(f"Adversarial (p={adv_prob:.2f})")
        axes[1].axis("off")

        diff = (denorm(adv) - denorm(clean))
        axes[2].imshow((diff - diff.min()) / (diff.max() - diff.min() + 1e-8))
        axes[2].set_title("Perturbation (normed)")
        axes[2].axis("off")
        plt.tight_layout()
        plt.show()


# Example usage on one FGSM batch
sample_batch = next(iter(valid_loader))
fgsm_result = fgsm_attack(sample_batch[0], sample_batch[1], epsilon=8/255)
plot_attack_examples(fgsm_result)

## 11. (Optional) Continue Training from the Loaded Checkpoint

If you wish to continue the 500-epoch run (or fine-tune further), you can reuse
the training loop below. It writes logs and checkpoints to the same directory
structure as before.

In [ ]:
from tqdm import tqdm


def train_one_epoch(epoch: int, global_step: int = 0) -> Tuple[float, float, float, float, int]:
    model.train()
    train_loss = 0.0
    train_correct = 0
    train_seen = 0
    for xb, yb in tqdm(train_loader, desc=f"Train epoch {epoch}"):
        xb = xb.to(DEVICE)
        yb = yb.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        preds = logits.argmax(dim=1)
        train_loss += loss.item() * yb.size(0)
        train_correct += (preds == yb).sum().item()
        train_seen += yb.size(0)
        global_step += 1
    train_loss /= max(1, train_seen)
    train_acc = train_correct / max(1, train_seen)

    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_seen = 0
    with torch.no_grad():
        for xb, yb in tqdm(valid_loader, desc="Validate"):
            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)
            logits = model(xb)
            loss = criterion(logits, yb)
            preds = logits.argmax(dim=1)
            val_loss += loss.item() * yb.size(0)
            val_correct += (preds == yb).sum().item()
            val_seen += yb.size(0)
    val_loss /= max(1, val_seen)
    val_acc = val_correct / max(1, val_seen)

    return train_loss, train_acc, val_loss, val_acc, global_step


def save_checkpoint(epoch: int, best_val: float, history: dict) -> None:
    state = {
        "epoch": epoch,
        "model_name": MODEL_NAME,
        "model_state": model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "scheduler_state": scheduler.state_dict() if scheduler is not None else None,
        "best_val": best_val,
        "history": history,
        "rng_state": get_rng_state(),
        "device_info": str(DEVICE),
    }
    tmp_path = LATEST_CKPT.with_suffix(".tmp")
    torch.save(state, tmp_path)
    os.replace(tmp_path, LATEST_CKPT)
    with open(HISTORY_JSON, "w", encoding="utf-8") as f:
        json.dump(history, f, indent=2)


history = defaultdict(list)
best_val = math.inf
start_epoch = 1
if resume_info is not None:
    start_epoch, best_val, loaded_history = resume_info
    for key, values in loaded_history.items():
        history[key] = list(values)

global_step = 0
# Example: run a single epoch (commented to avoid accidental execution).
# for epoch in range(start_epoch, start_epoch + 1):
#     tr_loss, tr_acc, va_loss, va_acc, global_step = train_one_epoch(epoch, global_step)
#     history["train_loss"].append(tr_loss)
#     history["train_acc"].append(tr_acc)
#     history["val_loss"].append(va_loss)
#     history["val_acc"].append(va_acc)
#     if scheduler is not None:
#         scheduler.step(va_loss)
#     if va_loss < best_val:
#         best_val = va_loss
#         torch.save(model.state_dict(), BEST_PATH)
#     save_checkpoint(epoch, best_val, history)

---

The notebook is now ready for interactive adversarial experimentation using the
previously trained ResNet-50 checkpoint without altering the established train
and validation splits.